In [60]:
import os
import csv
import time
import re
import string
import itertools
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from openai import OpenAI

# ----------------- CONFIG -----------------

TLD_FILE = "data/two_letter_tlds.txt"
OUTPUT_CSV = "domain_prices.csv"

ignored_tlds = {".bl", ".bq", ".eh"}

# Delay between OpenAI calls (seconds) to be nice to rate limits
OPENAI_SLEEP_SECONDS = 0.2

# Max characters of page text to send to GPT (controls cost)
MAX_TEXT_CHARS = 8000

# ------------------------------------------

client = OpenAI()


In [61]:
def load_tlds(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [
            line.strip().lower()
            for line in f
            if line.strip() and line.strip().lower() not in ignored_tlds
        ]


def generate_domains(tlds, length=1):
    base_chars = list(string.ascii_lowercase) + list(string.digits)

    if length == 1:
        labels = base_chars
    else:
        labels = ["".join(p) for p in itertools.product(base_chars, repeat=length)]

    for label in labels:
        for tld in tlds:
            yield f"{label}{tld}"


def fetch_page(domain: str):
    """
    Try HTTPS first, then HTTP. Returns (url, html_text) or (None, None).
    """
    headers = {
        "User-Agent": "DomainPriceChecker/1.0 (+https://example.com/your-contact)"
    }
    urls = [f"https://{domain}", f"http://{domain}"]

    for url in urls:
        try:
            resp = requests.get(url, timeout=10, headers=headers, allow_redirects=True)
            if resp.status_code == 200 and "text/html" in resp.headers.get(
                "Content-Type", ""
            ):
                return url, resp.text
        except requests.RequestException:
            # connection error / timeout / etc. -> try next URL
            continue

    return None, None


def extract_visible_text(html: str) -> str:
    """
    Strip scripts/styles and return a cleaned text string.
    """
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "header", "footer"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    text = re.sub(r"\s+", " ", text)  # collapse whitespace
    return text.strip()

def ensure_output_file(path: str):
    """
    Ensure CSV exists and has a header row.
    """
    file_exists = Path(path).is_file()
    if not file_exists:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["domain", "url", "price"])  # price can be NONE / NO_TEXT / NO_SITE



In [62]:
def ask_gpt_for_price(page_text: str) -> str:
    # Simple inline instruction prompt
    prompt = (
        "Tell me the price if there is one in this. "
        "Do not include commas, currency symbols, or words—"
        "only return a number. "
        "If there is no price return NONE. "
        f"{page_text}"
    )

    response = client.responses.create(
        model="gpt-4.1-nano",   # or gpt-5-nano if you prefer
        input=prompt,
        max_output_tokens=32,
    )

    # Print raw model output for debugging
    print("RAW GPT RESPONSE:", repr(response.output_text))

    raw = response.output_text
    if not raw:
        return "NONE"

    cleaned = (
        raw.replace("$", "")
           .replace(",", "")
           .replace("€", "")
           .replace("£", "")
           .strip()
    )

    # If GPT literally says NONE, trust it
    if cleaned.upper() == "NONE":
        return "NONE"

    # If it contains digits, assume it's a valid price
    if any(ch.isdigit() for ch in cleaned):
        return cleaned

    return "NONE"


In [63]:
tlds = load_tlds(TLD_FILE)
ensure_output_file(OUTPUT_CSV)

In [64]:

with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for idx, domain in enumerate(generate_domains(tlds), start=1):
        print(f"[{idx}] Checking {domain} ...")

        url, html = fetch_page(domain)
        if not url or not html:
            writer.writerow([domain, "", "NO_SITE"])
            f.flush()
            continue

        text = extract_visible_text(html)
        if not text:
            writer.writerow([domain, url, "NO_TEXT"])
            f.flush()
            continue

        price = ask_gpt_for_price(text)
        writer.writerow([domain, url, price])
        f.flush()

        print(f"    -> {price}")
        time.sleep(OPENAI_SLEEP_SECONDS)


[1] Checking a.ac ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[2] Checking a.ad ...
[3] Checking a.ae ...
[4] Checking a.af ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[5] Checking a.ag ...
[6] Checking a.ai ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[7] Checking a.al ...
[8] Checking a.am ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[9] Checking a.an ...
[10] Checking a.ao ...
[11] Checking a.aq ...
[12] Checking a.ar ...
RAW GPT RESPONSE: '100000'
    -> 100000
[13] Checking a.as ...
[14] Checking a.at ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[15] Checking a.au ...
[16] Checking a.aw ...
[17] Checking a.ax ...
[18] Checking a.az ...
[19] Checking a.ba ...
[20] Checking a.bb ...
[21] Checking a.bd ...
[22] Checking a.be ...
[23] Checking a.bf ...
[24] Checking a.bg ...
[25] Checking a.bh ...
[26] Checking a.bi ...
[27] Checking a.bj ...
[28] Checking a.bm ...
[29] Checking a.bn ...
[30] Checking a.bo ...
RAW GPT RESPONSE: 'NONE'
    -> NONE
[31] Checking a.br ...
[32] Checking a.bs ...
[33] Check

KeyboardInterrupt: 